# Spike Naive Comparison

Compare joint multidms fitting to naive independent-condition fits.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import time
import pickle
from functools import reduce

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import multidms

from notebooks._common import load_config, combine_replicate_muts, build_fit_params, robust_fit_models

In [ ]:
cfg = load_config(config_path, profile)
spike = cfg["spike"]
seed = cfg["seed"]
fit = spike["fitting"]

output_dir = spike["output_dir"]
reference = spike["reference"]
condition_colors = spike["condition_colors"]
chosen_lasso_strength = spike["chosen_lasso_strength"]
times_seen_threshold = spike["times_seen_threshold"]
gpu_ids = cfg["gpu_ids"]
n_processes = cfg["n_processes"]

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
%matplotlib inline
plt.rcParams.update({"legend.frameon": False, "font.size": 11})

In [ ]:
func_score_df = pd.read_csv(f"{output_dir}/training_functional_scores.csv").fillna({"aa_substitutions": ""})
models = pickle.load(open(f"{output_dir}/full_models.pkl", "rb"))

subsample_frac = spike.get("subsample_frac", None)
if subsample_frac is not None:
    func_score_df = (
        pd.concat([
        g.sample(frac=subsample_frac, random_state=seed)
        for _, g in func_score_df.groupby(["condition", "replicate"])
    ]).reset_index(drop=True)
    )
print(f"Loaded {len(func_score_df)} variants, {len(models)} models")

## Fit single-condition (naive) models

In [ ]:
cc = condition_colors
single_condition_datasets = []
for (replicate, condition), condition_fsdf in func_score_df.groupby(["replicate", "condition"]):
    df_agg = (
        condition_fsdf.groupby(["condition", "aa_substitutions"], dropna=False)
        .agg({"func_score": "mean"})
        .reset_index()
    )
    df_agg["aa_substitutions"] = df_agg["aa_substitutions"].fillna("")
    data = multidms.Data(
        df_agg,
        alphabet=multidms.AAS_WITHSTOP_WITHGAP,
        reference=condition,
        assert_site_integrity=False,
        verbose=False,
        name=f"{replicate}-{condition}",
    )
    data.condition_colors = cc
    single_condition_datasets.append(data)

single_condition_fit_params = build_fit_params(fit, single_condition_datasets)
single_condition_fit_params["fusionreg"] = [chosen_lasso_strength]

_, _, naive_models = robust_fit_models(
    single_condition_fit_params, gpu_ids=gpu_ids, n_processes=n_processes
)
for col in naive_models.columns:
    if naive_models[col].apply(lambda x: isinstance(x, dict)).any():
        naive_models[col] = naive_models[col].apply(str)

print(f"Fit {len(naive_models)} naive models")

## Compute naive shifts

In [ ]:
fit_dict = {row.dataset_name: row.model for _, row in naive_models.iterrows()}
naive_mut_df = combine_replicate_muts(fit_dict, how="inner", times_seen_threshold=times_seen_threshold)

# Compute naive shifts (difference of betas between conditions)
for condition in ["Delta", "Omicron_BA2"]:
    for replicate in [1, 2]:
        reference_betas = naive_mut_df[f"{replicate}-{reference}_beta_{reference}"]
        condition_betas = naive_mut_df[f"{replicate}-{condition}_beta_{condition}"]
        naive_mut_df[f"{replicate}-{condition}_S"] = condition_betas - reference_betas

print(f"Naive mutations: {len(naive_mut_df)} rows")

## Joint vs naive comparison plot

In [ ]:
# Get joint model mutations for comparison
mut_df_replicates = combine_replicate_muts(
    {
        f"{fit.dataset_name}".split("-")[-1]: fit.model
        for fit in models.query(f"fusionreg == {chosen_lasso_strength}").itertuples()
    },
    predicted_func_scores=False,
    how="inner",
    times_seen_threshold=times_seen_threshold,
)
mut_df_replicates["sense"] = [
    "stop" if mut == "*" else "nonsynonymous" for mut in mut_df_replicates.muts
]

saveas = "shift_distribution_correlation_naive"
pal = sns.color_palette("colorblind")

fig, axes = plt.subplots(2, 3, figsize=[9, 6])

# Row 1: Joint model replicate correlations
data = mut_df_replicates.dropna().copy()
for col, param in enumerate(["beta_Omicron_BA1", "shift_Delta", "shift_Omicron_BA2"]):
    iter_ax = axes[0, col]
    x, y = data[f"1_{param}"], data[f"2_{param}"]
    iter_ax.scatter(x, y, alpha=0.3, s=10, c="0.25")
    lim = [-6, 3] if col == 0 else [-3, 3]
    iter_ax.plot(lim, lim, "--", lw=2, c="royalblue")
    iter_ax.set_xlim(lim)
    iter_ax.set_ylim(lim)
    corr = pearsonr(x, y)[0] ** 2
    iter_ax.annotate(f"$R^2 = {corr:.2f}$", (0.07, 0.85), xycoords="axes fraction", fontsize=11)
    iter_ax.set_xlabel("replicate 1")
    if col == 0:
        iter_ax.set_ylabel("replicate 2")
    title = "mut. effect" if col == 0 else param.replace("shift_", "shift ")
    iter_ax.set_title(f"Joint: {title}", size=10)
    sns.despine(ax=iter_ax)

# Row 2: Naive model replicate correlations
data_naive = naive_mut_df.dropna().copy()
naive_params = [
    ("Omicron_BA1_beta_Omicron_BA1", "mut. effect"),
    ("Delta_S", "shift Delta"),
    ("Omicron_BA2_S", "shift BA.2"),
]
for col, (param, title) in enumerate(naive_params):
    iter_ax = axes[1, col]
    x = data_naive[f"1-{param}"]
    y = data_naive[f"2-{param}"]
    iter_ax.scatter(x, y, alpha=0.3, s=10, c="0.25")
    lim_lo = min(x.min(), y.min()) * 1.1
    lim_hi = max(x.max(), y.max()) * 1.1
    iter_ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], "--", lw=2, c="royalblue")
    corr = pearsonr(x, y)[0] ** 2
    iter_ax.annotate(f"$R^2 = {corr:.2f}$", (0.07, 0.85), xycoords="axes fraction", fontsize=11)
    iter_ax.set_xlabel("replicate 1")
    if col == 0:
        iter_ax.set_ylabel("replicate 2")
    iter_ax.set_title(f"Naive: {title}", size=10)
    sns.despine(ax=iter_ax)

plt.tight_layout()
fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()